# Notebook 01 — Data Exploration

Explore the raw data before building the Knowledge Graph.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingestion.demographics_ingestion import SAMPLE_DISTRICTS
from src.ingestion.healthcare_ingestion import SAMPLE_HOSPITALS

df_demo = pd.DataFrame(SAMPLE_DISTRICTS)
df_fac  = pd.DataFrame(SAMPLE_HOSPITALS)

print(f'Districts: {len(df_demo)}')
print(f'Facilities: {len(df_fac)}')

In [ ]:
# GP-per-1000 distribution
df_demo['gp_per_1000'] = df_demo['gp_count'] / df_demo['population'] * 1000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_demo.groupby('state')['gp_per_1000'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Mean GP-per-1000 by State')
axes[0].axvline(0.6, color='red', linestyle='--', label='Deficit threshold')
axes[0].legend()

df_demo.groupby('state')['age_65plus_pct'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Mean % Age 65+ by State')

plt.tight_layout()
plt.savefig('../data/processed/exploration_gp_age.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['population','age_65plus_pct','age_0_4_pct',
                'gp_per_1000','car_ownership_pct','median_income']

plt.figure(figsize=(8,6))
sns.heatmap(df_demo[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='RdYlGn', center=0)
plt.title('Demographic Variable Correlations')
plt.tight_layout()
plt.savefig('../data/processed/exploration_corr.png', dpi=150)
plt.show()

In [ ]:
# Facility type distribution
type_counts = df_fac['type'].value_counts()
type_counts.plot(kind='bar', color=['#e74c3c','#3498db','#2ecc71'])
plt.title('Healthcare Facility Types')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Map of facilities using folium
import folium

m = folium.Map(location=[47.8, 13.5], zoom_start=7)

colors = {'Hospital': 'red', 'GeneralPractitioner': 'blue', 'Pharmacy': 'green'}
for _, row in df_fac.iterrows():
    if pd.notna(row['lat']):
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=8,
            color=colors.get(row['type'], 'gray'),
            fill=True,
            popup=f"{row['name']} ({row['type']})"
        ).add_to(m)

m.save('../data/processed/facilities_map.html')
print('Map saved to data/processed/facilities_map.html')
m